[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S26_series_de_tiempo.ipynb)

# Sesión 26 · Series de tiempo

**Módulo 6: Negocio y extras** · ⏱️ Duración estimada: 60 a 75 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Construir una serie con índice de fechas y detectar días faltantes.
2. Cambiar de frecuencia con `resample` (de días a meses y años).
3. Suavizar el ruido con medias móviles (`rolling`).
4. Medir la estacionalidad semanal y anual, y descomponer una serie en tendencia, estacionalidad y residuo.
5. Evaluar pronósticos simples con un período de prueba.

## 📋 Qué debes saber antes
Sesión 12 (fechas en pandas) y sesiones 14 y 15 (gráficos de líneas).

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`, con los nombres de variables que se piden.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Genera los datos, aplica el estilo de gráficos y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión, aplica el estilo de gráficos y carga los verificadores.
import copy
import hashlib
import math
import statistics

import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Estilo de los gráficos ----------
# Paleta categórica en orden fijo (validada para daltonismo) y tintas para textos y ejes.
AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
GRIS = "#c3c2b7"          # para lo que no es protagonista
TINTA = "#0b0b0b"         # textos principales
TINTA_2 = "#52514e"       # textos secundarios
FONDO = "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TINTA_2, "text.color": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e1e0d9", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, AQUA, AMARILLO, MAGENTA, VERDE, VIOLETA, ROJO]),
    "axes.titlesize": 13, "axes.titlelocation": "left", "axes.titleweight": "bold",
    "lines.linewidth": 2, "font.size": 11, "figure.dpi": 100,
    "axes.formatter.useoffset": False, "axes.formatter.limits": (-9, 9),   # sin notación científica (1e6)
})



# ---------- Datos de práctica: ventas diarias de una tienda, 2022 a 2024 ----------
_fechas = pd.date_range("2022-01-01", "2024-12-31", freq="D")
_t = np.arange(len(_fechas))
_semana = np.array([0.85, 0.9, 0.92, 0.95, 1.1, 1.35, 1.2])[_fechas.dayofweek]
_mes = np.array([1.0, 0.9, 0.95, 0.97, 1.0, 1.0, 1.15, 1.0, 0.97, 1.0, 1.05, 1.35])[_fechas.month - 1]
_valor = (5000 + 1.2 * _t) * _semana * _mes * rng.normal(1, 0.06, len(_fechas))
_cerrado = ((_fechas.month == 1) & (_fechas.day == 1)) | ((_fechas.month == 5) & (_fechas.day == 1)) \
    | ((_fechas >= "2023-06-14") & (_fechas <= "2023-06-16"))
ventas_diarias = pd.DataFrame({"fecha": _fechas[~_cerrado].strftime("%Y-%m-%d"), "ventas": np.round(_valor[~_cerrado], 2)})
ventas_diarias = ventas_diarias.sample(frac=1, random_state=7).reset_index(drop=True)
DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]

_D = copy.deepcopy({"ventas_diarias": ventas_diarias})

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")

def _hex(c):
    return mcolors.to_hex(c).lower()


def _texto(a, b):
    return isinstance(a, str) and a.strip().lower() == b.strip().lower()


def _grafico(r, nombre):
    ax = r.var(nombre)
    if ax is _FALTA:
        return None
    if not isinstance(ax, mpl.axes.Axes):
        r.mal(f"`{nombre}` es de tipo {type(ax).__name__} y se esperaba un eje de Matplotlib (lo que devuelve `plt.subplots()`).")
        return None
    return ax


def _rotulos(r, nombre, ax, titulo=None, xlabel=None, ylabel=None):
    titulo_actual = next((t for t in (ax.get_title(loc=l) for l in ("left", "center", "right")) if t.strip()), "")
    for que, genero, obtenido, esperado in (("el título", "correcto", titulo_actual, titulo),
                                            ("la etiqueta del eje x", "correcta", ax.get_xlabel(), xlabel),
                                            ("la etiqueta del eje y", "correcta", ax.get_ylabel(), ylabel)):
        if esperado is None:
            continue
        if _texto(obtenido, esperado):
            r.ok(f"En `{nombre}`, {que} es {genero}.")
        elif not obtenido.strip():
            r.mal(f"A `{nombre}` le falta {que}.")
        else:
            r.mal(f"En `{nombre}`, {que} dice {obtenido!r}; revisa el texto pedido.")


def _barras(ax):
    """Rectángulos de barras (sin el fondo del eje), en orden de dibujo."""
    return [p for p in ax.patches if isinstance(p, mpl.patches.Rectangle)]


def _cerca_lista(a, b, tol=1e-6):
    return len(a) == len(b) and all(abs(float(x) - float(y)) <= tol for x, y in zip(a, b))


def _formato(ax, eje, valor):
    fmt = (ax.yaxis if eje == "y" else ax.xaxis).get_major_formatter()
    return fmt(valor, 0)


def _dias_ref():
    """Serie diaria completa recalculada con un diccionario (días sin venta = 0)."""
    valores = {pd.Timestamp(f): v for f, v in zip(ventas_diarias["fecha"], ventas_diarias["ventas"])}
    inicio, fin = min(valores), max(valores)
    dias, d = [], inicio
    while d <= fin:
        dias.append((d, valores.get(d, 0.0)))
        d += pd.Timedelta(days=1)
    return dias


def _mensual_ref():
    tot = {}
    for d, v in _dias_ref():
        clave = pd.Timestamp(d.year, d.month, 1)
        tot[clave] = tot.get(clave, 0.0) + v
    return tot


def _es_serie_temporal(r, nombre, v):
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series.")
        return False
    if not isinstance(v.index, pd.DatetimeIndex):
        r.mal(f"El índice de `{nombre}` debería ser de fechas (`DatetimeIndex`); usa `pd.to_datetime`.")
        return False
    return True


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    _sin_cambios_df(r, "ventas_diarias")
    dias = _dias_ref()
    abiertos = [(d, v) for d, v in dias if v != 0.0]
    s = r.var("serie")
    if s is not _FALTA and _es_serie_temporal(r, "serie", s):
        if not s.index.is_monotonic_increasing:
            r.mal("`serie` debería estar ordenada por fecha (`sort_index()`).")
        elif len(s) != len(abiertos) or [str(i) for i in s.index] != [str(d) for d, _ in abiertos] or not _cerca_lista(s.tolist(), [v for _, v in abiertos]):
            r.mal(f"`serie` debería tener las {len(abiertos)} ventas de `ventas_diarias`, con la fecha como índice.")
        else:
            r.ok("`serie` es correcta.")
    _esc(r, "n_faltantes", len(dias) - len(abiertos), "cuántos días del rango completo no tienen registro", tol=0)
    sc = r.var("serie_completa")
    if sc is not _FALTA and _es_serie_temporal(r, "serie_completa", sc):
        if len(sc) != len(dias):
            r.mal(f"`serie_completa` tiene {len(sc)} días y se esperaban {len(dias)}: uno por cada día entre la primera y la última fecha.")
        elif sc.isna().any():
            r.mal("`serie_completa` tiene vacíos: los días sin registro la tienda estuvo cerrada, así que sus ventas son 0.")
        elif not _cerca_lista(sc.tolist(), [v for _, v in dias]):
            r.mal("Los valores de `serie_completa` no coinciden con `serie` en los días con registro.")
        else:
            r.ok("`serie_completa` es correcta.")
    _esc(r, "total_marzo_2023", math.fsum(v for d, v in dias if d.year == 2023 and d.month == 3), "la suma de las ventas de marzo de 2023", tol=1e-6)
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_dias_2024": "d140e24b303fa78753a2500f28d1e63c1431d702695c2471e120e56df67323a2",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2 · Parte A")
    tot = _mensual_ref()
    claves = sorted(tot)
    _ser(r, "mensual", [tot[k] for k in claves], "la suma de `serie_completa` por mes, con el primer día de cada mes como índice", indice=[str(k) for k in claves], tol=1e-4)
    mejor = max(claves, key=lambda k: tot[k])
    v = r.var("mejor_mes")
    if v is not _FALTA:
        if str(v)[:10] != str(mejor)[:10]:
            r.mal("`mejor_mes` debería ser la fecha (índice) del mes con más ventas.")
        else:
            r.ok("`mejor_mes` es correcto.")
    anual = {a: math.fsum(tot[k] for k in claves if k.year == a) for a in (2022, 2023, 2024)}
    _ser(r, "anual", [anual[a] for a in (2022, 2023, 2024)], "la suma por año, con el primer día de cada año como índice",
         indice=[str(pd.Timestamp(a, 1, 1)) for a in (2022, 2023, 2024)], tol=1e-4)
    _esc(r, "crecimiento_2024", anual[2024] / anual[2023] - 1, "las ventas de 2024 divididas por las de 2023, menos 1", tol=1e-9)
    ax = _grafico(r, "ax_mensual")
    if ax is not None:
        lineas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(claves)]
        if not any(_cerca_lista(l.get_ydata(), [tot[k] for k in claves], 1e-4) for l in lineas):
            r.mal("`ax_mensual` debería tener una línea con las ventas de cada mes.")
        else:
            r.ok("`ax_mensual` muestra las ventas mensuales.")
        _rotulos(r, "ax_mensual", ax, None, None, "Ventas (S/)")
    r.fin()
    r = _Revision("Ejercicio 2 · Parte B")
    r.predicciones({
        "pred_etiqueta_ms": "ee7c91bc078f21dc0f52f8cc3e99aac952ffca864b0a49d9797f84d914773cec",
    })
    r.fin()


def _movil(valores, n):
    return [None if i < n - 1 else math.fsum(valores[i - n + 1:i + 1]) / n for i in range(len(valores))]


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    dias = _dias_ref()
    vals = [v for _, v in dias]
    for nombre, n in (("media_7", 7), ("media_28", 28)):
        v = r.var(nombre)
        if v is _FALTA or not _es_serie_temporal(r, nombre, v):
            continue
        ref = _movil(vals, n)
        if len(v) != len(ref):
            r.mal(f"`{nombre}` tiene {len(v)} valores y se esperaban {len(ref)}: calcúlala sobre `serie_completa`.")
        elif not all(_mismo(a, b, 1e-6) for a, b in zip(v.tolist(), ref)):
            r.mal(f"Los valores de `{nombre}` no coinciden: es el promedio de los últimos {n} días, incluido el actual (`rolling({n}).mean()`).")
        else:
            r.ok(f"`{nombre}` es correcta.")
    ax = _grafico(r, "ax_suave")
    if ax is not None:
        lineas = [l for l in ax.get_lines() if len(l.get_ydata()) == len(dias)]
        diaria = [l for l in lineas if _cerca_lista(l.get_ydata(), vals, 1e-6)]
        suave = [l for l in lineas if all(_mismo(float(a) if not np.isnan(a) else None, b, 1e-6) for a, b in zip(np.asarray(l.get_ydata(), dtype=float), _movil(vals, 28)))]
        if not diaria or not suave:
            r.mal("`ax_suave` debería tener dos líneas: las ventas diarias y la media móvil de 28 días.")
        elif _hex(diaria[0].get_color()) != GRIS or _hex(suave[0].get_color()) != AZUL:
            r.mal("En `ax_suave`, deja las ventas diarias en `GRIS` (contexto) y la media móvil en `AZUL` (protagonista).")
        elif ax.get_legend() is None or len(ax.get_legend().get_texts()) != 2:
            r.mal("Agrega una leyenda con las dos líneas a `ax_suave`.")
        else:
            r.ok("`ax_suave` destaca la tendencia sobre el ruido diario.")
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_nan_media_7": "4152dd8442a25c19608edaff98259df5e9c2e72140ea93426a877bca387c65b6",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    dias = _dias_ref()
    por_dia = [statistics.fmean([v for d, v in dias if d.dayofweek == i]) for i in range(7)]
    _ser(r, "por_dia", por_dia, "el promedio de `serie_completa` por día de la semana, con `DIAS` como índice", indice=DIAS, tol=1e-6)
    v = r.var("dia_mas_fuerte", str)
    if v is not _FALTA:
        if v.strip() != DIAS[por_dia.index(max(por_dia))]:
            r.mal("`dia_mas_fuerte` debería ser el nombre (de `DIAS`) del día con mayor promedio.")
        else:
            r.ok("`dia_mas_fuerte` es correcto.")
    tot = _mensual_ref()
    media = statistics.fmean(tot.values())
    indice = [statistics.fmean([x for k, x in tot.items() if k.month == m]) / media for m in range(1, 13)]
    _ser(r, "indice_mes", indice, "el promedio de `mensual` por mes del año dividido por el promedio de `mensual`, con los meses 1 a 12 como índice", indice=list(range(1, 13)), tol=1e-9)
    _esc(r, "mes_mas_fuerte", indice.index(max(indice)) + 1, "el número del mes con mayor índice", tol=0)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_diciembre": "5c3d07cddad1d8ddb78e3bc88af34426d03c871c28e2924590cf32c22c551832",
    })
    r.fin()


def _descomp_ref():
    tot = _mensual_ref()
    claves = sorted(tot)
    y = [tot[k] for k in claves]
    tend = [None] * len(y)
    for i in range(len(y)):
        ini, fin = i - 6, i + 6          # ventana centrada de 12: 6 antes y 5 después (así la centra pandas)
        if ini >= 0 and fin <= len(y):
            tend[i] = math.fsum(y[ini:fin]) / 12
    dif = {}
    for k, a, b in zip(claves, y, tend):
        if b is not None:
            dif.setdefault(k.month, []).append(a - b)
    est = {m: statistics.fmean(v) for m, v in dif.items()}
    prom = statistics.fmean(est.values())
    est = {m: x - prom for m, x in est.items()}
    resid = [None if b is None else a - b - est[k.month] for k, a, b in zip(claves, y, tend)]
    return claves, tend, est, resid


def check_ejercicio_5():
    r = _Revision("Ejercicio 5 · Parte A")
    claves, tend, est, resid = _descomp_ref()
    _ser(r, "tendencia", tend, "la media móvil centrada de 12 meses de `mensual` (`rolling(12, center=True).mean()`)", indice=[str(k) for k in claves], tol=1e-4)
    _ser(r, "estacional", [est[m] for m in range(1, 13)], "el promedio de `mensual - tendencia` por mes del año, menos el promedio de esos 12 valores, con los meses 1 a 12 como índice", indice=list(range(1, 13)), tol=1e-4)
    _ser(r, "residuo", resid, "`mensual - tendencia - la estacionalidad de su mes`", indice=[str(k) for k in claves], tol=1e-4)
    validos = [(k, abs(x)) for k, x in zip(claves, resid) if x is not None]
    v = r.var("mes_atipico")
    if v is not _FALTA:
        if str(v)[:10] != str(max(validos, key=lambda t: t[1])[0])[:10]:
            r.mal("`mes_atipico` debería ser la fecha del mes con el residuo más grande en valor absoluto.")
        else:
            r.ok("`mes_atipico` es correcto.")
    r.fin()


def _pronosticos_ref():
    tot = _mensual_ref()
    claves = sorted(tot)
    y = [tot[k] for k in claves]
    corte = claves.index(pd.Timestamp("2024-07-01"))
    tr, te = y[:corte], y[corte:]
    ingenuo = [tr[-1]] * len(te)
    est_ing = [y[corte + i - 12] for i in range(len(te))]
    n = len(tr)
    tm, ym = statistics.fmean(range(n)), statistics.fmean(tr)
    b = math.fsum((t - tm) * (v - ym) for t, v in zip(range(n), tr)) / math.fsum((t - tm) ** 2 for t in range(n))
    a = ym - b * tm
    res = {}
    for t, v in zip(range(n), tr):
        res.setdefault(claves[t].month, []).append(v - (a + b * t))
    est = {m: statistics.fmean(x) for m, x in res.items()}
    tend_est = [a + b * (corte + i) + est[claves[corte + i].month] for i in range(len(te))]
    return claves[corte:], te, {"ingenuo": ingenuo, "estacional_ingenuo": est_ing, "tendencia_estacional": tend_est}


def check_reto():
    r = _Revision("Reto final")
    fechas, real, pron = _pronosticos_ref()
    cols = ["real", "ingenuo", "estacional_ingenuo", "tendencia_estacional"]
    filas = [[real[i]] + [pron[c][i] for c in cols[1:]] for i in range(len(real))]
    _df(r, "pronosticos", cols, filas, "una fila por mes de prueba (julio a diciembre de 2024) con el valor real y los tres pronósticos", indice=[str(f) for f in fechas], tol=1e-3)
    mae = {c: statistics.fmean([abs(p - v) for p, v in zip(pron[c], real)]) for c in cols[1:]}
    _ser(r, "errores", [mae[c] for c in cols[1:]], "el error absoluto medio de cada método, con los nombres de los métodos como índice", indice=cols[1:], tol=1e-3)
    v = r.var("mejor_metodo", str)
    if v is not _FALTA:
        if v.strip() != min(mae, key=mae.get):
            r.mal("`mejor_metodo` debería ser el nombre del método con menor error.")
        else:
            r.ok("`mejor_metodo` es correcto.")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    fechas, real, _ = _pronosticos_ref()
    p = r.var("pronostico_hw")
    if p is not _FALTA:
        p = np.asarray(p, dtype=float)
        if p.shape != (len(real),):
            r.mal(f"`pronostico_hw` tiene forma {p.shape} y se esperaba ({len(real)},): un valor por mes de prueba.")
        elif not np.isfinite(p).all() or any(not 0.6 * v < x < 1.4 * v for x, v in zip(p, real)):
            r.mal("Los valores de `pronostico_hw` no parecen un pronóstico de estas ventas: ajusta el modelo solo con `entreno` y usa `forecast(6)`.")
        elif abs(p[5] / p[4] - 1) < 0.1:
            r.mal("`pronostico_hw` no muestra el salto de diciembre: revisa `seasonal=\"add\"` y `seasonal_periods=12`.")
        else:
            r.ok("`pronostico_hw` tiene la forma y el patrón esperados.")
            _esc(r, "mae_hw", statistics.fmean([abs(x - v) for x, v in zip(p, real)]), "el error absoluto medio entre `pronostico_hw` y las ventas reales de la prueba", tol=1e-6)
    r.fin()


print("✅ Setup listo. Datos generados, estilo aplicado y verificadores cargados.")

### 📦 Tus datos de hoy
`ventas_diarias`: las ventas (en soles) de una tienda, día por día, de 2022 a 2024. Vienen **desordenadas** y con la fecha como texto. Los días en que la tienda cerró no tienen fila. `DIAS` tiene los nombres de los días de la semana, de lunes a domingo.

In [ ]:
print(ventas_diarias.head(), "\n")
print(ventas_diarias.dtypes, "\n")
print(len(ventas_diarias), ventas_diarias["fecha"].min(), ventas_diarias["fecha"].max())

---
## 1. El índice temporal

### 📘 Concepto
Una **serie de tiempo** es una Series con un índice de fechas (`DatetimeIndex`). Con ese índice, pandas entiende el calendario:
- `serie.loc["2023-03"]` selecciona todo marzo de 2023; `serie.loc["2023"]`, todo el año;
- `serie.sort_index()` ordena por fecha;
- `serie.asfreq("D")` crea un índice con **todos** los días entre la primera y la última fecha; los días sin dato quedan como `NaN`.

Antes de analizar, decide qué significa un día faltante: si la tienda cerró, las ventas fueron 0 (`fillna(0)`); si el dato se perdió, no.

In [ ]:
s_ej = pd.Series([10.0, 12.0, 9.0], index=pd.to_datetime(["2024-01-03", "2024-01-01", "2024-01-05"])).sort_index()
print(s_ej, "\n")
print(s_ej.asfreq("D"), "\n")
print(s_ej.loc["2024-01-01":"2024-01-03"].sum())

### ✍️ Tu turno · Ejercicio 1: armar la serie
**Parte A.**
1. `serie`: una Series con las ventas, la fecha convertida con `pd.to_datetime` como índice, ordenada por fecha.
2. `n_faltantes`: cuántos días entre la primera y la última fecha no tienen registro.
3. `serie_completa`: `serie` con todos los días, con 0 en los días en que la tienda cerró.
4. `total_marzo_2023`: las ventas totales de marzo de 2023.

**Parte B.** Predice **sin ejecutar**: ¿cuántos días de 2024 tiene `serie_completa`? Guárdalo en `pred_dias_2024` (un entero).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

`ventas_diarias.set_index(pd.to_datetime(ventas_diarias["fecha"]))["ventas"]` y luego `.sort_index()`.
</details>

<details><summary>💡 Pista 2</summary>

`n_faltantes = serie.asfreq("D").isna().sum()` y `serie_completa = serie.asfreq("D").fillna(0)`.
</details>

---
## 2. Cambiar de frecuencia: `resample`

### 📘 Concepto
`resample` es un `groupby` por períodos de tiempo. Se elige la frecuencia y luego cómo resumir:
- `serie.resample("MS").sum()`: total por mes, etiquetado con el **primer** día del mes (*month start*);
- `serie.resample("W").mean()`: promedio por semana (que termina en domingo);
- `serie.resample("YS").sum()`: total por año.

Para ventas, la suma tiene sentido; para un saldo o una temperatura, el promedio o el último valor.

In [ ]:
dias_ej = pd.Series(range(1, 61), index=pd.date_range("2024-01-01", periods=60, freq="D"))
print(dias_ej.resample("MS").sum())
print(dias_ej.resample("MS").mean())

### ✍️ Tu turno · Ejercicio 2: de días a meses y años
**Parte A.**
1. `mensual`: las ventas totales por mes de `serie_completa`.
2. `mejor_mes`: la fecha (índice) del mes con más ventas.
3. `anual`: las ventas totales por año, y `crecimiento_2024`: cuánto crecieron las ventas de 2024 respecto de 2023, como proporción (0.1 = 10 %).
4. `fig_mensual, ax_mensual`: una línea con `mensual`, eje y `Ventas (S/)` y un título-conclusión.

¿Ves un patrón que se repite cada año?

**Parte B.** Responde en `pred_etiqueta_ms` con `"primero"` o `"último"`: ¿con qué día del mes etiqueta `resample("MS")` cada mes?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

`mensual = serie_completa.resample("MS").sum()` y `mensual.idxmax()`.
</details>

<details><summary>💡 Pista 2</summary>

`crecimiento_2024 = anual.loc["2024"].iloc[0] / anual.loc["2023"].iloc[0] - 1`, o usa `anual.iloc[2] / anual.iloc[1] - 1`.
</details>

---
## 3. Suavizar con medias móviles: `rolling`

### 📘 Concepto
Las ventas diarias tienen mucho ruido. Una **media móvil** promedia los últimos `n` días en cada fecha:
- `serie.rolling(7).mean()`: promedio de los 7 días que terminan en cada fecha; elimina el patrón semanal;
- los primeros `n − 1` valores quedan en `NaN`, porque todavía no hay `n` días.

En el gráfico, la serie original va de contexto (en gris, fina) y la media móvil de protagonista.

In [ ]:
print(pd.Series([1, 2, 3, 4, 5, 6]).rolling(3).mean().tolist())

### ✍️ Tu turno · Ejercicio 3: ver la tendencia a través del ruido
**Parte A.**
1. `media_7` y `media_28`: las medias móviles de 7 y 28 días de `serie_completa`.
2. `fig_suave, ax_suave`: las ventas diarias en `GRIS` con `linewidth=0.6` y la media de 28 días en `AZUL`, con leyenda y un título-conclusión.

**Parte B.** Predice **sin ejecutar**: ¿cuántos `NaN` tiene `media_7` al inicio? Guárdalo en `pred_nan_media_7` (un entero).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

Dibuja las dos líneas en el mismo eje con `ax_suave.plot(indice, valores, color=..., label=...)`.
</details>

<details><summary>💡 Pista 2</summary>

Dibuja primero la diaria (queda atrás) y después la media móvil; termina con `ax_suave.legend()`.
</details>

---
## 4. Estacionalidad

### 📘 Concepto
La **estacionalidad** es un patrón que se repite con un período fijo: más ventas los sábados, en julio (gratificaciones) o en diciembre. Se mide agrupando por la posición dentro del período:
- día de la semana: `serie.index.dayofweek` (0 = lunes, 6 = domingo);
- mes del año: `serie.index.month` (1 a 12).

Un **índice estacional** divide el promedio de cada mes por el promedio general: 1,2 significa un mes 20 % por encima de lo normal. Así se separa lo que es "época del año" de lo que es crecimiento.

In [ ]:
print(dias_ej.groupby(dias_ej.index.dayofweek).mean())

### ✍️ Tu turno · Ejercicio 4: ¿cuándo se vende más?
**Parte A.**
1. `por_dia`: el promedio de `serie_completa` por día de la semana, con `DIAS` como índice (de lunes a domingo).
2. `dia_mas_fuerte`: el nombre del día con mayor promedio.
3. `indice_mes`: el promedio de `mensual` por mes del año (índice 1 a 12) dividido por el promedio de `mensual`.
4. `mes_mas_fuerte`: el número del mes con mayor índice.

**Parte B.** Responde en `pred_diciembre` con `"tendencia"` o `"estacionalidad"`: que diciembre venda más que noviembre todos los años, ¿es tendencia o estacionalidad?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

`serie_completa.groupby(serie_completa.index.dayofweek).mean()` da un índice 0 a 6: cámbialo por `DIAS`.
</details>

<details><summary>💡 Pista 2</summary>

`indice_mes = mensual.groupby(mensual.index.month).mean() / mensual.mean()`.
</details>

---
## 5. Descomposición: tendencia + estacionalidad + residuo

### 📘 Concepto
Una serie mensual se puede pensar como la suma de tres partes (**descomposición aditiva**):

`ventas = tendencia + estacionalidad + residuo`

1. **Tendencia**: media móvil **centrada** de 12 meses (`rolling(12, center=True).mean()`); promediar un año entero borra la estacionalidad. Los extremos quedan en `NaN`.
2. **Estacionalidad**: el promedio de `ventas − tendencia` para cada mes del año, ajustado para que las 12 cifras sumen 0 (se les resta su promedio).
3. **Residuo**: lo que sobra. Un residuo grande marca un mes raro que merece una explicación.

In [ ]:
ondas_ej = pd.Series(np.arange(36) * 2 + np.tile([5, -5, 0], 12), index=pd.date_range("2022-01-01", periods=36, freq="MS"))
tend_ej = ondas_ej.rolling(3, center=True).mean()
print(pd.DataFrame({"serie": ondas_ej, "tendencia": tend_ej}).head(6))

### ✍️ Tu turno · Ejercicio 5: separar las partes
**Parte A.**
1. `tendencia`: la media móvil centrada de 12 meses de `mensual`.
2. `estacional`: el promedio de `mensual - tendencia` por mes del año (índice 1 a 12), menos el promedio de esos 12 valores.
3. `residuo`: `mensual - tendencia` menos la estacionalidad de cada mes (con el mismo índice que `mensual`).
4. `mes_atipico`: la fecha del mes con el residuo más grande en valor absoluto.

Opcional: grafica las cuatro series en `plt.subplots(4, 1, sharex=True)`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

`(mensual - tendencia).groupby(mensual.index.month).mean()` ignora los `NaN` de los extremos.
</details>

<details><summary>💡 Pista 2</summary>

Para restar la estacionalidad de cada mes: `estacional.reindex(mensual.index.month).to_numpy()`.
</details>

---
## 🏋️ Reto final: ¿qué pronóstico usarías?
Para evaluar un pronóstico se esconde el final de la serie: se pronostica con lo anterior y se compara con lo que pasó. Usa `mensual` hasta junio de 2024 como entrenamiento y julio a diciembre de 2024 como prueba.
1. `pronosticos`: un DataFrame con los 6 meses de prueba como índice y las columnas:
   - `real`: las ventas reales;
   - `ingenuo`: el último mes de entrenamiento repetido;
   - `estacional_ingenuo`: el valor del mismo mes del año anterior;
   - `tendencia_estacional`: una recta ajustada al entrenamiento con `np.polyfit(t, y, 1)` (con `t = 0, 1, 2...` como número de mes) más el promedio de los residuos de esa recta para cada mes del año.
2. `errores`: una Series con el error absoluto medio (MAE) de cada método (índice con los tres nombres, en ese orden).
3. `mejor_metodo`: el nombre del método con menor error.

¿Por qué el método ingenuo falla tanto en diciembre?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

`entreno = mensual.loc[:"2024-06"]` y `prueba = mensual.loc["2024-07":]`. El estacional ingenuo es `mensual.shift(12).loc["2024-07":]`.
</details>

<details><summary>💡 Pista 2</summary>

Con `a, b = np.polyfit(t, entreno.to_numpy(), 1)` la recta es `a * t + b`. Los residuos son `entreno - recta`, y su promedio por `entreno.index.month` da el ajuste de cada mes.
</details>

---
## 🚀 Nivel pro (opcional): Holt-Winters con statsmodels
`statsmodels` trae modelos clásicos de series de tiempo. Ajusta `ExponentialSmoothing(entreno, trend="add", seasonal="add", seasonal_periods=12).fit(method="least_squares")` (de `statsmodels.tsa.holtwinters`) con los meses de entrenamiento del reto (con frecuencia mensual: `entreno.asfreq("MS")`). Guarda `pronostico_hw` (el pronóstico de los 6 meses de prueba, como array) y `mae_hw` (su error absoluto medio). ¿Le gana a tus métodos simples?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P4 · la evolución en el tiempo

**Qué hacer**
1. Si tus datos tienen fecha, arma una serie mensual (o trimestral) de reclamos: total y, si te sirve, por producto o por segmento de S24.
2. Revisa los meses faltantes y decide qué significan; anótalo.
3. Grafica la serie con una media móvil y marca los meses atípicos con una anotación que los explique, si encuentras la causa (un cambio de norma, una campaña, una caída de un sistema).
4. Normaliza cuando compares con el crecimiento del crédito (pregunta 4 del proyecto): reclamos por cada 10 000 clientes o índices de base 100, en un solo eje.
5. Si quieres proyectar, usa un método simple y evaluado con un período de prueba, y preséntalo como escenario, no como certeza.

**Por qué lo haría un analista**
Quien decide pregunta "¿está mejorando o empeorando?". Separar la tendencia de la estacionalidad evita celebrar (o alarmarse) por un mes que siempre es alto.

**Cómo debe verse el resultado**
Un gráfico de evolución con título-conclusión, una tabla con la variación anual y, si corresponde, un pronóstico con su error en prueba. Irán a tu dashboard y a tu presentación final (S27).

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Convertir una columna de texto en un índice de fechas y ordenar la serie.
- [ ] Encontrar y tratar días faltantes con `asfreq`.
- [ ] Usar `resample` para pasar de días a semanas, meses y años.
- [ ] Explicar qué hace una media móvil y por qué tiene `NaN` al inicio.
- [ ] Medir la estacionalidad semanal y anual y separar tendencia, estacionalidad y residuo.
- [ ] Evaluar pronósticos con un período de prueba y compararlos con el MAE.

**Próxima sesión (S27):** comunicar resultados: propuesta de negocio, slides para decisores, notebook limpio y referencias. Es la última del plan.